In [ ]:
!pip install yfinance requests

In [ ]:
!pip install openpyxl

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import csv
import time
import datetime
import requests
from pathlib import Path

sys.path.append(os.path.abspath(".."))
from src.ETF_holdings_enriched import process_universal_etf
from dotenv import load_dotenv
# Load environment variables
load_dotenv()

In [ ]:
%run ../src/portfolio_engine.py

In [ ]:
# input paths/ETF's and output path + file name from .env file -- please also check description.txt for more details
base_dir = Path(os.getenv("data_dir", "."))
output_dir = os.path.join(base_dir, "processed")
etf1_output_csv = Path(output_dir) / "ETF1_holdings_enriched.csv"
etf2_output_csv = Path(output_dir) / "ETF2_holdings_enriched.csv"
etf3_output_csv = Path(output_dir) / "ETF3_holdings_enriched.csv"
etf4_output_csv = Path(output_dir) / "ETF4_holdings_enriched.csv"
input_dir = Path(os.getenv("ETF1_path"))
etf1_input_csv = Path(os.getenv("ETF1"))
etf2_input_csv = Path(os.getenv("ETF2"))
etf3_input_csv = Path(os.getenv("ETF3"))
etf4_input_csv = Path(os.getenv("ETF4"))
etf1_map_file = Path(input_dir) / "name_ticker_manual_map.csv"

In [ ]:
# Configuration for Source 1
index_ticker = os.getenv("etf1_ticker")
index_name = os.getenv("etf1_name")
etf1_config = {
    "index_name": index_name,      
    "index_ticker": index_ticker,             
    "col_name": "Component Name",                
    "col_isin": None,                
    "col_weight": "Weight %", 
    "col_country": None
}

# Configuration for Source 2
index_ticker = os.getenv("etf2_ticker")
index_name = os.getenv("etf2_name")
etf2_config = {
    "index_name": index_name,      
    "index_ticker": index_ticker,             
    "col_name": "Name",                
    "col_isin": "ISIN",                
    "col_weight": "Weighting", 
    "col_country": "Country"
}

# Configuration for Source 3
index_ticker = os.getenv("etf3_ticker")
index_name = os.getenv("etf3_name")
etf3_config = {
    "index_name": index_name,      
    "index_ticker": index_ticker,             
    "col_name": "Name",                
    "col_isin": "ISIN",                
    "col_weight": "Weighting", 
    "col_country": "Country"
}

# Configuration for Source 4
index_ticker = os.getenv("etf4_ticker")
index_name = os.getenv("etf4_name")
Country = os.getenv("etf4_country")
etf4_config = {
    "index_name": index_name,      
    "index_ticker": index_ticker,             
    "col_name": "Security Name",                
    "col_isin": "ISIN",                
    "col_weight": "Weight (%)", 
    "col_country": Country
}

In [ ]:
def run_update_orchestrator():
    # 1. Automated 6-Month Check using local file
    date_file = "last_run.txt"
    today = datetime.datetime.now()
    
    if os.path.exists(date_file):
        with open(date_file, "r") as f:
            last_date_str = f.read().strip()
            last_update = datetime.datetime.strptime(last_date_str, "%Y-%m-%d")
    else:
        # If file doesn't exist, set a default past date to trigger the alert
        last_update = today - datetime.timedelta(days=200)
    
    if (today - last_update).days >= 180:
        print("!!! ALERT: It has been 6 months since your last update.")
        print("Please check the ETF providers' websites for new holding files.\n")

    # 2. Define your configurations mapping
    etf_registry = {
        '1': {'config': etf1_config, 'input': etf1_input_csv, 'output': etf1_output_csv, 'map': etf1_map_file},
        '2': {'config': etf2_config, 'input': etf2_input_csv, 'output': etf2_output_csv, 'map': None},
        '3': {'config': etf3_config, 'input': etf3_input_csv, 'output': etf3_output_csv, 'map': None},
        '4': {'config': etf4_config, 'input': etf4_input_csv, 'output': etf4_output_csv, 'map': None}
    }

    # 3. Ask the user for input
    print("Which ETF holdings have you updated? (Enter numbers, e.g., 2,4)")
    print("Type 'q' to quit (no updates today). or Enter 'all' to run all updates.")
    choice = input("Your choice: ").strip().lower()

    if choice == 'q':
        print("No updates selected. Exiting.")
        return

    # 4. Determine which to run
    targets = etf_registry.keys() if choice == 'all' else [c.strip() for c in choice.split(',')]

    # 5. Execute selected targets
    ran_update = False
    for t in targets:
        if t in etf_registry:
            data = etf_registry[t]
            print(f"\n--- Running update for ETF {t} ---")
            process_universal_etf(
                input_path=data['input'],
                output_path=data['output'],
                map_path=data['map'],
                config=data['config']
            )
            ran_update = True
        else:
            print(f"Skipping: ETF {t} is not a valid selection.")
            
    # 6. Update the tracker file if updates were performed
    if ran_update:
        with open(date_file, "w") as f:
            f.write(today.strftime("%Y-%m-%d"))
        print("\nTracker updated. Next 6-month check starts from today.")

run_update_orchestrator() 